# Benchmark Performance

Computes per-similarity-bin Pearson r for one or more models against the NFAB test set.

**Inputs** (configure in the next cell):
- `ACTIVITIES_PATH`: path to `activities.parquet` produced by `nfab_preprocess.run_pipeline`
- `MODELS`: dict mapping a display name to a predictions CSV path

Predictions CSVs must have columns: `assay_id`, `ligand_name`, `standard_type`, `pred_pchembl`.
Activity rows without a matching prediction are filled with `pred_pchembl = 6.0` (1 µM).

In [ ]:
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nfab_evaluate.plotting import _parse_split_label, load_predictions, compute_model_stats

ACTIVITIES_PATH = pathlib.Path("out/activities.parquet")

# Map model display name → path to predictions CSV.
# The CSV must have columns: assay_id, ligand_name, standard_type, pred_pchembl.
MODELS: dict[str, str | pathlib.Path] = {
    "Ligand-only FP baseline": "out_baseline/predictions_v0.csv"
}

MIN_ASSAY_SIZE = 10  # minimum compounds per (assay, standard_type) to include in Pearson r
N_BOOTSTRAP = 1000   # bootstrap resamples for standard error

## Load activities

In [ ]:
activities = pd.read_parquet(
    ACTIVITIES_PATH,
    columns=["assay_chembl_id", "ligand_chembl_id", "standard_type", "split", "pchembl_value_filled"],
)

test_activities = activities[activities["split"].str.startswith("test_sim_", na=False)].copy()
print(f"Total test rows: {len(test_activities):,}")
print("Splits found:")
for s, n in sorted(test_activities["split"].value_counts().items()):
    print(f"  {s}: {n:,}")

## Discover bins from split labels

In [ ]:
unique_splits = sorted(
    test_activities["split"].unique(),
    key=lambda s: _parse_split_label(s)[0],
)
bins = [_parse_split_label(s) for s in unique_splits]  # list of (lo, hi, label)

print("Bins (in plot order):")
for lo, hi, label in bins:
    n = (test_activities["split"] == (f"test_sim_{lo:.2f}" if lo == hi else f"test_sim_{lo:.2f}_{hi:.2f}")).sum()
    print(f"  {label}: {n:,} rows")

## Load predictions and compute per-bin Pearson r

In [ ]:
# tol:vibrant qualitative palette (colorblind-safe)
_TOL_VIBRANT = [
    "#0077BB",  # blue
    "#EE7733",  # orange
    "#009988",  # teal
    "#CC3311",  # red
    "#33BBEE",  # cyan
    "#EE3377",  # magenta
    "#BBBBBB",  # grey
]

all_stats: dict[str, pd.DataFrame] = {}

for model_name, pred_path in MODELS.items():
    print(f"\nLoading predictions for: {model_name}")
    merged = load_predictions(pred_path, test_activities)
    stats = compute_model_stats(merged, bins, min_assay_size=MIN_ASSAY_SIZE, n_bootstrap=N_BOOTSTRAP)
    all_stats[model_name] = stats
    print(stats.to_string(index=False))

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Use n_m / n_a from the first model for x-axis labels (same across models)
first_stats = next(iter(all_stats.values()))
x = np.arange(len(first_stats))

for i, (model_name, stats) in enumerate(all_stats.items()):
    color = _TOL_VIBRANT[i % len(_TOL_VIBRANT)]
    r = stats["pearson_r"].values
    ci_low = stats["ci_low"].values
    ci_high = stats["ci_high"].values

    ax.plot(
        x, r,
        marker="o",
        markersize=8,
        linewidth=2,
        color=color,
        label=model_name,
    )
    ax.fill_between(
        x,
        ci_low,
        ci_high,
        alpha=0.15,
        color=color,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    [
        f"{row.display_label}\n$n_m$={row.n_m:,}\n$n_a$={row.n_a}"
        for row in first_stats.itertuples()
    ],
    fontsize=10,
)

ax.set_xlabel(
    "Max Tanimoto similarity to train and val compounds",
    fontsize=12,
    fontweight="bold",
)
ax.set_ylabel("Mean Pearson Correlation ($r$)", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.tick_params(axis="y", labelsize=11)
ax.legend(frameon=False, fontsize=11, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()